# 01 — Nettoyage des données logistics

**Groupe 3 — Siyaman MOHAMAD ; autres membres à compléter.**

**Objectif :** produire sept tables exploitables à partir des CSV bruts, en suivant le CM1.
Les valeurs atypiques restent dans les données. Ce notebook prépare les données ; les analyses
et interprétations se trouvent dans `02_analyse_logistics.ipynb`.

**Ordre d’utilisation :** exécuter ce notebook depuis le début, puis ouvrir le second avec un noyau neuf.
Les deux notebooks sont à la racine du projet et utilisent l’environnement `datamining`.

Les données et le récit sont fictifs.


## 1. Charger les fichiers et établir le bilan initial

Le dictionnaire `data/data_dictionary.csv` sert à identifier le sens des colonnes.
Les identifiants restent des identifiants, même lorsqu’une partie de leur contenu est numérique.
Les fichiers `data/logistics/*.csv` ne sont pas modifiés.

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
dossier = Path("data/logistics")

tables_brutes = {
    fichier.stem: pd.read_csv(fichier)
    for fichier in sorted(dossier.glob("*.csv"))
}

bilan = pd.DataFrame([
    {
        "table": nom,
        "lignes": len(df),
        "colonnes": df.shape[1],
        "doublons_exacts": df.duplicated().sum(),
        "cellules_manquantes": df.isna().sum().sum()
    }
    for nom, df in tables_brutes.items()
])

display(bilan)

,table,lignes,colonnes,doublons_exacts,cellules_manquantes
0,erp_business_partners,80,11,0,15
1,erp_facilities,30,11,0,1
2,goods_receipts,612,12,2,4
3,inventory_adjustments,140,11,0,221
4,inventory_month_end,502,11,2,0
5,purchase_order_lines,651,16,1,3
6,shipments,452,16,2,17


## 2. Harmoniser les textes et retirer les doublons exacts

On retire les espaces en début et fin et on utilise les minuscules dans les sept tables,
y compris dans les codes, afin de garder une convention commune pour les futurs croisements.
Les espaces internes et les valeurs manquantes sont conservés. Les chaînes vides deviennent manquantes.

In [3]:
tables_nettoyees = {}

for nom, df in tables_brutes.items():
    propre = df.copy()

    for col in propre.select_dtypes(include=["object", "string"]):
        propre[col] = propre[col].str.strip().str.lower().replace("", pd.NA)

    propre = propre.drop_duplicates().reset_index(drop=True)
    tables_nettoyees[nom] = propre

    print(nom, ":", len(df) - len(propre), "doublon(s) supprimé(s)")

erp_business_partners : 0 doublon(s) supprimé(s)
erp_facilities : 0 doublon(s) supprimé(s)
goods_receipts : 2 doublon(s) supprimé(s)
inventory_adjustments : 0 doublon(s) supprimé(s)
inventory_month_end : 2 doublon(s) supprimé(s)
purchase_order_lines : 1 doublon(s) supprimé(s)
shipments : 2 doublon(s) supprimé(s)


## 3. Fonctions de conversion utilisées dans chaque fichier

Les dates observées utilisent les formats ci-dessous. La fonction essaie ces formats et signale
les valeurs qu’elle ne comprend pas. Pour les formes jour/mois/année, nous retenons la convention
française ; les valeurs ambiguës ne peuvent pas être départagées par leur texte seul.

Pour les quantités, on remplace la virgule décimale et on conserve le nombre situé avant l’unité.
**Avant de retirer un suffixe, on vérifie sa concordance avec la colonne d’unité.**

In [4]:
# Formats observés dans les données.
formats = [
    "%Y-%m-%d",           # 2022-10-01
    "%Y/%m/%d",           # 2022/10/01
    "%d/%m/%Y",           # 26/04/2018
    "%d-%b-%Y",           # 10-apr-2023
    "%Y-%m-%d %H:%M:%S",  # 2023-04-13 06:00:00
    "%Y/%m/%d %H:%M:%S",  # 2022/02/10 13:00:00
    "%d/%m/%Y %H:%M:%S",  # 16/02/2018 12:00:00
    "%d-%b-%Y %H:%M:%S"   # 15-may-2021 06:00:00
]

def convertir_date(valeur):
    if pd.isna(valeur):
        return pd.NaT  # Conserver les dates absentes.

    # Remplacer le séparateur "t" ou "T" entre la date et l'heure.
    texte = str(valeur)
    if len(texte) > 10 and texte[10] in ["t", "T"]:
        texte = texte[:10] + " " + texte[11:]

    for fmt in formats:
        try:
            return pd.to_datetime(texte, format=fmt, errors="raise")
        except ValueError:
            continue  # Essayer le format suivant.

    raise ValueError(f"Date non reconnue : {valeur}")

In [5]:
def convertir_nombre(colonne):
    texte = colonne.astype("string").str.replace(",", ".", regex=False)
    return pd.to_numeric(texte.str.split().str[0], errors="raise")

### 3.1. `erp_business_partners.csv`

Une ligne décrit un partenaire commercial. `partner_code` identifie le partenaire.

**Valeurs manquantes :** Les numéros d’enregistrement et régions absents sont conservés : les autres champs ne permettent pas de les reconstituer avec certitude.

In [6]:
partenaires = tables_nettoyees["erp_business_partners"]

display(partenaires.head())
manquants = partenaires.isna().sum()
display(manquants[manquants > 0].to_frame("nombre_manquant"))

,partner_code,partner_name,partner_role,external_registration_number,billing_region,shipping_region,default_currency,payment_terms,partner_status,risk_class,onboarding_date
0,bp-30001,marula cold store cooperative,supplier_customer,reg-avl-20012,avelora federation,eastern avelora agricultural cluster,avl,net45,active,review,2001-02-01
1,bp-30002,blue heron monitoring cooperative,customer,reg-knt-20037,kintara republic,kintara karst cluster,krn,net45,active,standard,2012-03-01
2,bp-30003,orra technical services,supplier,reg-avl-20022,avelora federation,eastern avelora agricultural cluster,avl,public30,active,review,2008-04-01
3,bp-30004,common ground climate forum,service_partner,reg-vln-01008,valenne republic,valenne central district,vln,prepaid,active,review,2014-05-01
4,bp-30005,warden medical ventures,service_partner,reg-vln-01016,valenne republic,valenne republic,vln,prepaid,inactive,standard,2013-06-01


,nombre_manquant
external_registration_number,14
billing_region,1


In [7]:
# Observer les dates qui ne suivent pas le format habituel.
for col in ['onboarding_date']:
    texte = partenaires[col].astype("string")
    format_attendu = "%Y-%m-%d"
    if col in ["dispatch_time", "arrival_time"]:
        format_attendu = "%Y-%m-%dt%H:%M:%S"
    test = pd.to_datetime(texte, format=format_attendu, errors="coerce")
    autres_formats = texte[texte.notna() & test.isna()]
    if not autres_formats.empty:
        print(col)
        display(autres_formats.drop_duplicates().to_frame())

onboarding_date


,onboarding_date
20,01/10/2003


In [8]:
for col in ['onboarding_date']:
    partenaires[col] = partenaires[col].apply(convertir_date)

display(partenaires.dtypes.to_frame("type_apres_conversion"))

,type_apres_conversion
partner_code,str
partner_name,str
partner_role,str
external_registration_number,str
billing_region,str
shipping_region,str
default_currency,str
payment_terms,str
partner_status,str
risk_class,str


### 3.2. `erp_facilities.csv`

Une ligne décrit un site. `facility_code` identifie le site et `operating_partner_code` désigne son opérateur.

**Valeurs manquantes :** Une capacité de stockage non renseignée reste inconnue. Les coordonnées sont déjà numériques.

In [9]:
sites = tables_nettoyees["erp_facilities"]

display(sites.head())
manquants = sites.isna().sum()
display(manquants[manquants > 0].to_frame("nombre_manquant"))

,facility_code,facility_name,facility_category,operating_partner_code,street_address,region,latitude,longitude,opening_date,facility_status,storage_capabilities
0,fcl-1011,meridian valenne headquarters,office,bp-30078,"meridian valenne headquarters, valenne central...",valenne central district,47.24,7.58,2006-01-01,active,ambient
1,fcl-1024,meridian kintara biological aggregation facility,warehouse,bp-30078,meridian kintara biological aggregation facili...,kintara karst cluster,4.74,112.28,1997-01-01,active,ambient;secure_cage;2-8c;frozen
2,fcl-1019,meridian avelora transfer annex,transfer_hub,bp-30078,"meridian avelora transfer annex, eastern avelo...",eastern avelora agricultural cluster,7.46,101.91,2003-01-01,active,ambient;secure_cage;2-8c;frozen
3,fcl-1020,meridian tembara coastal hub,transfer_hub,bp-30078,"meridian tembara coastal hub, tembara coastal ...",tembara coastal cluster,2.20,104.55,2010-01-01,active,ambient;secure_cage;2-8c;frozen
4,fcl-1016,northstar tembara cold-chain hub,warehouse,bp-30047,"northstar tembara cold-chain hub, tembara coas...",tembara coastal cluster,2.23,104.58,2008-01-01,active,ambient;secure_cage;2-8c;frozen


,nombre_manquant
storage_capabilities,1


In [10]:
# Observer les dates qui ne suivent pas le format habituel.
for col in ['opening_date']:
    texte = sites[col].astype("string")
    format_attendu = "%Y-%m-%d"
    if col in ["dispatch_time", "arrival_time"]:
        format_attendu = "%Y-%m-%dt%H:%M:%S"
    test = pd.to_datetime(texte, format=format_attendu, errors="coerce")
    autres_formats = texte[texte.notna() & test.isna()]
    if not autres_formats.empty:
        print(col)
        display(autres_formats.drop_duplicates().to_frame())

In [11]:
for col in ['opening_date']:
    sites[col] = sites[col].apply(convertir_date)

display(sites.dtypes.to_frame("type_apres_conversion"))

,type_apres_conversion
facility_code,str
facility_name,str
facility_category,str
operating_partner_code,str
street_address,str
region,str
latitude,float64
longitude,float64
opening_date,datetime64[us]
facility_status,str


### 3.3. `purchase_order_lines.csv`

Une ligne décrit une ligne de commande. La clé est le couple `purchase_order_number`, `purchase_order_line`.

**Valeurs manquantes :** Les centres de coût absents sont conservés. Les prix restent associés à leur devise et à leur unité.

In [12]:
commandes = tables_nettoyees["purchase_order_lines"]

display(commandes.head())
manquants = commandes.isna().sum()
display(manquants[manquants > 0].to_frame("nombre_manquant"))

,purchase_order_number,purchase_order_line,order_date,ordering_facility_code,supplier_partner_code,item_code,item_description,item_category,ordered_quantity,ordered_unit,unit_price,currency,requested_delivery_date,authorization_class,declared_cost_center,order_status
0,po-2014-00001,1,2014-06-24,fcl-1007,bp-30078,itm-med-001,general clinical consumable case,medical_consumables,73,case,157.85,vln,2014-07-08,restricted,clinical programmes,completed
1,po-2015-00001,1,2015-01-22,fcl-1010,bp-30068,itm-trn-001,regional refrigerated transport service,transport_service,1,service,3482.27,vln,2015-02-01,ordinary,clinical services,completed
2,po-2015-00002,1,2015-01-29,fcl-1030,bp-30066,itm-com-001,secure communication service,communication_service,1,service,1292.19,krn,2015-02-26,ordinary,general operations,completed
3,po-2015-00003,1,2015-02-10,fcl-1010,bp-30080,itm-lab-002,sample preparation material pack,laboratory_material,3,pack,458.67,vln,2015-03-02,expedited,clinical services,completed
4,po-2015-00004,1,2015-02-12,fcl-1022,bp-30021,itm-gra-001,programme grant transfer,grant,1,grant,26933.28,krn,2015-02-21,ordinary,facilities,completed


,nombre_manquant
declared_cost_center,3


In [13]:
# Observer les dates qui ne suivent pas le format habituel.
for col in ['order_date', 'requested_delivery_date']:
    texte = commandes[col].astype("string")
    format_attendu = "%Y-%m-%d"
    if col in ["dispatch_time", "arrival_time"]:
        format_attendu = "%Y-%m-%dt%H:%M:%S"
    test = pd.to_datetime(texte, format=format_attendu, errors="coerce")
    autres_formats = texte[texte.notna() & test.isna()]
    if not autres_formats.empty:
        print(col)
        display(autres_formats.drop_duplicates().to_frame())

order_date


,order_date
384,2022/10/01
446,10-apr-2023
521,2023/10/23


requested_delivery_date


,requested_delivery_date
101,26/04/2018
148,29-jul-2019
346,2022/02/17


In [14]:
# Observer les nombres contenant une virgule ou du texte.
for col in ['ordered_quantity', 'unit_price']:
    texte = commandes[col].astype("string")
    problemes = texte.str.contains(r"[,a-zA-Z_]", na=False)
    if problemes.any():
        print(col)
        display(texte[problemes].drop_duplicates().to_frame())

ordered_quantity


,ordered_quantity
87,88 case
100,1 service
176,24 pack
183,18 unit
306,1 contract
333,30 pallet
457,12 kit


unit_price


,unit_price
107,"2469,48"
129,"2474,89"
159,"563,83"
173,"4239,29"
240,"560,35"
253,"4045,45"


In [15]:
# Vérifier que l’unité intégrée est bien conservée dans sa colonne dédiée.
colonnes_unites = {'ordered_quantity': 'ordered_unit'}
for col, col_unite in colonnes_unites.items():
    texte = commandes[col].astype("string")
    unite_texte = texte.str.split(n=1).str[1]
    conflit = unite_texte.notna() & unite_texte.ne(commandes[col_unite]).fillna(True)
    if conflit.any():
        display(commandes.loc[conflit, [col, col_unite]])
        raise ValueError("Unité à vérifier avant conversion")
print("Aucun conflit d’unité détecté.")

Aucun conflit d’unité détecté.


In [16]:
for col in ['order_date', 'requested_delivery_date']:
    commandes[col] = commandes[col].apply(convertir_date)

for col in ['ordered_quantity', 'unit_price']:
    commandes[col] = convertir_nombre(commandes[col])

display(commandes.dtypes.to_frame("type_apres_conversion"))

,type_apres_conversion
purchase_order_number,str
purchase_order_line,int64
order_date,datetime64[us]
ordering_facility_code,str
supplier_partner_code,str
item_code,str
item_description,str
item_category,str
ordered_quantity,int64
ordered_unit,str


### 3.4. `goods_receipts.csv`

Une ligne décrit une réception. `receipt_number` identifie la réception ; le numéro de commande et son numéro de ligne permettent un rapprochement ultérieur.

**Valeurs manquantes :** Les quatre statuts d’inspection absents ne sont pas remplacés par le statut majoritaire. `posted` ne prouve pas une inspection acceptée.

In [17]:
receptions = tables_nettoyees["goods_receipts"]

display(receptions.head())
manquants = receptions.isna().sum()
display(manquants[manquants > 0].to_frame("nombre_manquant"))

,receipt_number,purchase_order_number,purchase_order_line,receipt_date,receiving_facility_code,supplier_partner_code,received_quantity,received_unit,batch_reference,inspection_status,storage_location_code,receipt_status
0,rcv-2014-00001,po-2014-00001,1,2014-07-08,fcl-1007,bp-30078,73,case,b-2014-44545,held,bin-35,quality_hold
1,rcv-2015-00002,po-2015-00001,1,2015-02-02,fcl-1010,bp-30068,0.9,service,b-2015-58495,accepted,bin-36,posted
2,rcv-2015-00003,po-2015-00002,1,2015-02-28,fcl-1030,bp-30066,1,service,b-2015-48474,held,bin-17,quality_hold
3,rcv-2015-00004,po-2015-00003,1,2015-03-02,fcl-1010,bp-30080,3,pack,b-2015-71913,accepted,bin-16,posted
4,rcv-2015-00005,po-2015-00004,1,2015-02-21,fcl-1022,bp-30021,1,grant,b-2015-31154,accepted,bin-42,posted


,nombre_manquant
inspection_status,4


In [18]:
# Observer les dates qui ne suivent pas le format habituel.
for col in ['receipt_date']:
    texte = receptions[col].astype("string")
    format_attendu = "%Y-%m-%d"
    if col in ["dispatch_time", "arrival_time"]:
        format_attendu = "%Y-%m-%dt%H:%M:%S"
    test = pd.to_datetime(texte, format=format_attendu, errors="coerce")
    autres_formats = texte[texte.notna() & test.isna()]
    if not autres_formats.empty:
        print(col)
        display(autres_formats.drop_duplicates().to_frame())

receipt_date


,receipt_date
224,2020/05/02
442,2023/09/03
527,12/07/2024
568,2025/06/11
609,30/06/2025


In [19]:
# Observer les nombres contenant une virgule ou du texte.
for col in ['received_quantity']:
    texte = receptions[col].astype("string")
    problemes = texte.str.contains(r"[,a-zA-Z_]", na=False)
    if problemes.any():
        print(col)
        display(texte[problemes].drop_duplicates().to_frame())

received_quantity


,received_quantity
27,"56,41"
52,"0,88"
56,14 kit
87,"69,37"
302,28 case
326,36 pack
333,21 kit
387,"2,56"
417,4 kit
422,"35,65"


In [20]:
# Vérifier que l’unité intégrée est bien conservée dans sa colonne dédiée.
colonnes_unites = {'received_quantity': 'received_unit'}
for col, col_unite in colonnes_unites.items():
    texte = receptions[col].astype("string")
    unite_texte = texte.str.split(n=1).str[1]
    conflit = unite_texte.notna() & unite_texte.ne(receptions[col_unite]).fillna(True)
    if conflit.any():
        display(receptions.loc[conflit, [col, col_unite]])
        raise ValueError("Unité à vérifier avant conversion")
print("Aucun conflit d’unité détecté.")

Aucun conflit d’unité détecté.


In [21]:
for col in ['receipt_date']:
    receptions[col] = receptions[col].apply(convertir_date)

for col in ['received_quantity']:
    receptions[col] = convertir_nombre(receptions[col])

display(receptions.dtypes.to_frame("type_apres_conversion"))

,type_apres_conversion
receipt_number,str
purchase_order_number,str
purchase_order_line,int64
receipt_date,datetime64[us]
receiving_facility_code,str
supplier_partner_code,str
received_quantity,float64
received_unit,str
batch_reference,str
inspection_status,str


### 3.5. `shipments.csv`

Une ligne décrit une étape d’expédition. On teste la clé `shipment_number`, `shipment_leg`.

**Valeurs manquantes :** Les dates d’arrivée absentes sont conservées. On vérifie leur relation avec le statut avant d’interpréter leur absence. `declared_quantity` est déjà numérique.

In [22]:
expeditions = tables_nettoyees["shipments"]

display(expeditions.head())
manquants = expeditions.isna().sum()
display(manquants[manquants > 0].to_frame("nombre_manquant"))

,shipment_number,shipment_leg,dispatch_time,arrival_time,sender_partner_code,receiver_partner_code,origin_facility_code,destination_facility_code,carrier_partner_code,item_category,declared_quantity,declared_unit,temperature_control_class,authorization_class,declared_purpose,shipment_status
0,shp-2014-00001,1,2014-07-10t09:00:00,2014-07-11t03:00:00,bp-30078,bp-30021,fcl-1011,fcl-1007,bp-30047,biological_sample,2,sealed_lot,2-8c,expedited,temperature-controlled clinical programme deli...,delivered
1,shp-2015-00001,1,2015-01-10t11:00:00,2015-01-13t16:00:00,bp-30035,bp-30033,fcl-1009,fcl-1013,bp-30020,medical_consumables,73,case,2-8c,ordinary,scheduled stock transfer,returned
2,shp-2015-00002,1,2015-01-11t06:00:00,2015-01-12t00:00:00,bp-30028,bp-30037,fcl-1020,fcl-1017,bp-30015,office_supply,82,case,ambient,emergency,customer distribution,delivered
3,shp-2015-00003,1,2015-02-17t15:00:00,2015-02-19t08:00:00,bp-30056,bp-30078,fcl-1019,fcl-1011,bp-30047,biological_sample,2,sealed_lot,2-8c,restricted,laboratory delivery,delivered
4,shp-2015-00004,1,2015-06-10t12:00:00,2015-06-12t09:00:00,bp-30025,bp-30079,fcl-1026,fcl-1022,bp-30025,legal_service,1,service,ambient,ordinary,facility replenishment,delivered


,nombre_manquant
arrival_time,17


In [23]:
# Observer les dates qui ne suivent pas le format habituel.
for col in ['dispatch_time', 'arrival_time']:
    texte = expeditions[col].astype("string")
    format_attendu = "%Y-%m-%d"
    if col in ["dispatch_time", "arrival_time"]:
        format_attendu = "%Y-%m-%dt%H:%M:%S"
    test = pd.to_datetime(texte, format=format_attendu, errors="coerce")
    autres_formats = texte[texte.notna() & test.isna()]
    if not autres_formats.empty:
        print(col)
        display(autres_formats.drop_duplicates().to_frame())

dispatch_time


,dispatch_time
214,15-may-2021 06:00:00
247,2022/02/10 13:00:00
347,2023/09/09 08:00:00


arrival_time


,arrival_time
56,16/02/2018 12:00:00
109,11-aug-2019 18:00:00
410,07-jul-2024 22:00:00


In [24]:
for col in ['dispatch_time', 'arrival_time']:
    expeditions[col] = expeditions[col].apply(convertir_date)

display(expeditions.dtypes.to_frame("type_apres_conversion"))

,type_apres_conversion
shipment_number,str
shipment_leg,int64
dispatch_time,datetime64[us]
arrival_time,datetime64[us]
sender_partner_code,str
receiver_partner_code,str
origin_facility_code,str
destination_facility_code,str
carrier_partner_code,str
item_category,str


Vérification de contexte : les dates d’arrivée absentes concernent-elles des expéditions encore en transit ?

In [25]:
display(expeditions[expeditions["arrival_time"].isna()]["shipment_status"].value_counts())

shipment_status
in_transit    17
Name: count, dtype: int64

### 3.6. `inventory_adjustments.csv`

Une ligne décrit un ajustement de stock. `adjustment_number` identifie l’ajustement. Une quantité négative représente une diminution selon la convention examinée.

**Valeurs manquantes :** Les références de réception ou d’expédition absentes et le rôle d’approbation manquant sont conservés. Leur absence sera analysée selon le motif et le statut.

In [26]:
ajustements = tables_nettoyees["inventory_adjustments"]

display(ajustements.head())
manquants = ajustements.isna().sum()
display(manquants[manquants > 0].to_frame("nombre_manquant"))

,adjustment_number,facility_code,adjustment_date,item_category,adjustment_quantity,unit,adjustment_reason,approved_by_role,related_receipt_number,related_shipment_number,adjustment_status
0,adj-2014-00001,fcl-1007,2014-07-18,medical_consumables,-1.58,case,specialist programme issue,clinical stores manager,NaN,NaN,posted
1,adj-2015-00002,fcl-1007,2015-09-10,medical_consumables,-2.7,case,specialist programme issue,specialist programme administrator,NaN,NaN,posted
2,adj-2015-00003,fcl-1024,2015-10-30,monitoring_equipment,3.68,kit,late receipt posting,warehouse manager,rcv-2022-00361,NaN,posted
3,adj-2016-00004,fcl-1007,2016-01-20,biological_sample,6.52,sealed_lot,late receipt posting,finance controller,NaN,NaN,pending_review
4,adj-2016-00005,fcl-1010,2016-02-18,medical_consumables,-2.43,case,procedure stock issue,quality manager,NaN,NaN,posted


,nombre_manquant
approved_by_role,1
related_receipt_number,117
related_shipment_number,103


In [27]:
# Observer les dates qui ne suivent pas le format habituel.
for col in ['adjustment_date']:
    texte = ajustements[col].astype("string")
    format_attendu = "%Y-%m-%d"
    if col in ["dispatch_time", "arrival_time"]:
        format_attendu = "%Y-%m-%dt%H:%M:%S"
    test = pd.to_datetime(texte, format=format_attendu, errors="coerce")
    autres_formats = texte[texte.notna() & test.isna()]
    if not autres_formats.empty:
        print(col)
        display(autres_formats.drop_duplicates().to_frame())

adjustment_date


,adjustment_date
107,12/11/2023
115,21-mar-2024


In [28]:
# Observer les nombres contenant une virgule ou du texte.
for col in ['adjustment_quantity']:
    texte = ajustements[col].astype("string")
    problemes = texte.str.contains(r"[,a-zA-Z_]", na=False)
    if problemes.any():
        print(col)
        display(texte[problemes].drop_duplicates().to_frame())

adjustment_quantity


,adjustment_quantity
56,-2.9 case


In [29]:
# Vérifier que l’unité intégrée est bien conservée dans sa colonne dédiée.
colonnes_unites = {'adjustment_quantity': 'unit'}
for col, col_unite in colonnes_unites.items():
    texte = ajustements[col].astype("string")
    unite_texte = texte.str.split(n=1).str[1]
    conflit = unite_texte.notna() & unite_texte.ne(ajustements[col_unite]).fillna(True)
    if conflit.any():
        display(ajustements.loc[conflit, [col, col_unite]])
        raise ValueError("Unité à vérifier avant conversion")
print("Aucun conflit d’unité détecté.")

Aucun conflit d’unité détecté.


In [30]:
for col in ['adjustment_date']:
    ajustements[col] = ajustements[col].apply(convertir_date)

for col in ['adjustment_quantity']:
    ajustements[col] = convertir_nombre(ajustements[col])

display(ajustements.dtypes.to_frame("type_apres_conversion"))

,type_apres_conversion
adjustment_number,str
facility_code,str
adjustment_date,datetime64[us]
item_category,str
adjustment_quantity,float64
unit,str
adjustment_reason,str
approved_by_role,str
related_receipt_number,str
related_shipment_number,str


### 3.7. `inventory_month_end.csv`

Une ligne décrit un bilan de stock pour un site, un mois, une catégorie et une unité. Les stocks sont des niveaux : on ne les additionne pas sur plusieurs mois comme des flux.

**Valeurs manquantes :** Aucune valeur manquante n’a été repérée dans le bilan initial. Les décimales et les unités sont conservées. Un écart de bilan n’est pas corrigé sans justification.

In [31]:
stocks = tables_nettoyees["inventory_month_end"]

display(stocks.head())
manquants = stocks.isna().sum()
display(manquants[manquants > 0].to_frame("nombre_manquant"))

,facility_code,reporting_month,item_category,opening_stock,received_quantity,outgoing_quantity,documented_internal_use,documented_waste,closing_stock,unit,inventory_status
0,fcl-1024,2018-02-01,biological_sample,14.55 sealed_lot,5.46,0.00,0.49,0.22,16.24,sealed_lot,closed
1,fcl-1024,2018-03-01,biological_sample,16.24,5.00,0.00,1.94,0.65,0.55,sealed_lot,closed
2,fcl-1024,2018-05-01,biological_sample,0.55,0.00,0.00,0.00,0.36,0.17,sealed_lot,closed
3,fcl-1024,2019-01-01,biological_sample,0.17,0.00,1.37,0.00,0.02,0.00,sealed_lot,provisional
4,fcl-1024,2019-08-01,biological_sample,0.00,3.12,8.80,0.00,0.08,0.00,sealed_lot,closed


,nombre_manquant


In [32]:
# Observer les dates qui ne suivent pas le format habituel.
for col in ['reporting_month']:
    texte = stocks[col].astype("string")
    format_attendu = "%Y-%m-%d"
    if col in ["dispatch_time", "arrival_time"]:
        format_attendu = "%Y-%m-%dt%H:%M:%S"
    test = pd.to_datetime(texte, format=format_attendu, errors="coerce")
    autres_formats = texte[texte.notna() & test.isna()]
    if not autres_formats.empty:
        print(col)
        display(autres_formats.drop_duplicates().to_frame())

In [33]:
# Observer les nombres contenant une virgule ou du texte.
for col in ['opening_stock', 'received_quantity', 'outgoing_quantity', 'documented_internal_use', 'documented_waste', 'closing_stock']:
    texte = stocks[col].astype("string")
    problemes = texte.str.contains(r"[,a-zA-Z_]", na=False)
    if problemes.any():
        print(col)
        display(texte[problemes].drop_duplicates().to_frame())

opening_stock


,opening_stock
0,14.55 sealed_lot
55,22.79 kit
151,75.12 kit
335,270.62 case


received_quantity


,received_quantity
96,"1,04"
306,"5,42"
390,"5,51"


outgoing_quantity


,outgoing_quantity
190,"17,00"
307,"0,00"
368,"2,97"


documented_waste


,documented_waste
25,"1,45"
340,"1,29"


closing_stock


,closing_stock
17,0.92 sealed_lot
167,74.59 case
285,45.75 pallet
487,1.03 pallet


In [34]:
# Vérifier que l’unité intégrée est bien conservée dans sa colonne dédiée.
colonnes_unites = {'opening_stock': 'unit', 'received_quantity': 'unit', 'outgoing_quantity': 'unit', 'documented_internal_use': 'unit', 'documented_waste': 'unit', 'closing_stock': 'unit'}
for col, col_unite in colonnes_unites.items():
    texte = stocks[col].astype("string")
    unite_texte = texte.str.split(n=1).str[1]
    conflit = unite_texte.notna() & unite_texte.ne(stocks[col_unite]).fillna(True)
    if conflit.any():
        display(stocks.loc[conflit, [col, col_unite]])
        raise ValueError("Unité à vérifier avant conversion")
print("Aucun conflit d’unité détecté.")

Aucun conflit d’unité détecté.


In [35]:
for col in ['reporting_month']:
    stocks[col] = stocks[col].apply(convertir_date)

for col in ['opening_stock', 'received_quantity', 'outgoing_quantity', 'documented_internal_use', 'documented_waste', 'closing_stock']:
    stocks[col] = convertir_nombre(stocks[col])

display(stocks.dtypes.to_frame("type_apres_conversion"))

,type_apres_conversion
facility_code,str
reporting_month,datetime64[us]
item_category,str
opening_stock,float64
received_quantity,float64
outgoing_quantity,float64
documented_internal_use,float64
documented_waste,float64
closing_stock,float64
unit,str


## 4. Bilan final et contrôle des clés

On vérifie que les clés choisies sont complètes et uniques après conversion. Une clé répétée
ne serait pas supprimée automatiquement : il faudrait d’abord examiner les lignes concernées.
Le contrôle des clés ne garantit pas à lui seul la cohérence de tous les liens entre fichiers.

In [36]:
cles = {
    "erp_business_partners": ["partner_code"],
    "erp_facilities": ["facility_code"],
    "purchase_order_lines": ["purchase_order_number", "purchase_order_line"],
    "goods_receipts": ["receipt_number"],
    "shipments": ["shipment_number", "shipment_leg"],
    "inventory_adjustments": ["adjustment_number"],
    "inventory_month_end": ["facility_code", "reporting_month", "item_category", "unit"]
}

bilan_final = []
for nom, df in tables_nettoyees.items():
    bilan_final.append({
        "table": nom,
        "lignes_brutes": len(tables_brutes[nom]),
        "lignes_nettoyees": len(df),
        "cellules_manquantes": int(df.isna().sum().sum()),
        "cles_absentes": int(df[cles[nom]].isna().any(axis=1).sum()),
        "cles_repetees": int(df.duplicated(cles[nom], keep=False).sum())
    })

display(pd.DataFrame(bilan_final))

,table,lignes_brutes,lignes_nettoyees,cellules_manquantes,cles_absentes,cles_repetees
0,erp_business_partners,80,80,15,0,0
1,erp_facilities,30,30,1,0,0
2,goods_receipts,612,610,4,0,0
3,inventory_adjustments,140,140,221,0,0
4,inventory_month_end,502,500,0,0,0
5,purchase_order_lines,651,650,3,0,0
6,shipments,452,450,17,0,0


## 5. Enregistrer les sept tables nettoyées

`to_pickle()` enregistre une table pandas avec ses types (dates, nombres et valeurs manquantes).
Les fichiers `.pkl` produits ici sont lus avec `read_pickle()` dans le notebook d’analyse.
Cela évite de répéter les conversions lors du chargement. Aucun paquet supplémentaire n’est requis.
Le dossier de sortie est distinct du dossier des CSV bruts. Une nouvelle exécution met à jour ces sorties.

In [37]:
dossier_sortie = Path("data/logistics_nettoyees")
dossier_sortie.mkdir(exist_ok=True)

for nom, df in tables_nettoyees.items():
    df.to_pickle(dossier_sortie / f"{nom}.pkl")

print("Sept tables nettoyées enregistrées dans", dossier_sortie)

Sept tables nettoyées enregistrées dans data/logistics_nettoyees


**Bilan :** les formats ont été harmonisés et les doublons exacts retirés. Les quantités atypiques,
statuts rares et valeurs manquantes sont conservés. Ouvrir maintenant `02_analyse_logistics.ipynb`.